In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [19]:
# Prepare the 18 acoustic features and target labels from the Soundwel dataset.
# Train and evaluate the existing Logistic Regression and HistGradientBoosting
# baseline models using random, session-grouped, and leave-one-team-out
# cross-validation. These results will serve as a benchmark for comparing
# our additional SVM, Random Forest, and XGBoost models.

In [2]:
import os, hashlib, zipfile, subprocess, shutil
from pathlib import Path
from collections import Counter
import pandas as pd

DATA = Path("/kaggle/working/soundwel")
DATA.mkdir(parents=True, exist_ok=True)

FILES = {
    "soundwel.zip": (
        "https://zenodo.org/records/8252482/files/Soundwel%20Dataset%20-%20Audio%20and%20Spectrograms.zip?download=1",
        "dc2144fd683a7c9b27246d51817db9b4"),
    "SoundwelDatasetKey.xlsx": (
        "https://zenodo.org/records/8252482/files/SoundwelDatasetKey.xlsx?download=1",
        "2f87e52a9b28c18e570ac6aae5fb27b9"),
}

def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def is_good(path, expected):
    return path.exists() and path.stat().st_size > 0 and md5(path) == expected

# download only if missing OR corrupted; fail loudly if it still doesn't verify
for name, (url, expected) in FILES.items():
    out = DATA / name
    if not is_good(out, expected):
        out.unlink(missing_ok=True)
        print(f"Downloading {name} ...")
        r = subprocess.run(
            ["wget", "--tries=5", "--timeout=120", "--retry-connrefused",
             "--progress=dot:giga", "-O", str(out), url])
        if r.returncode != 0:
            out.unlink(missing_ok=True)
            raise RuntimeError(
                f"wget failed (exit {r.returncode}). Exit 4 / 'resolve host' = internet is off "
                "(Settings -> Internet). 403/429 = Zenodo is throttling; use Add Data instead.")
    got = md5(out)
    if got != expected:
        print(open(out, "rb").read(300))
        out.unlink(missing_ok=True)
        raise RuntimeError(f"{name}: md5 mismatch ({got}); deleted. Re-run to retry.")
    print(f"{name}: {out.stat().st_size / 1e6:.1f} MB, md5 OK")

# unzip into a temp folder, rename only on success
raw = DATA / "raw"
if not raw.exists():
    tmp = DATA / "raw_tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    with zipfile.ZipFile(DATA / "soundwel.zip") as z:
        z.extractall(tmp)
    tmp.rename(raw)

# ---- everything below is your original code ----
print("\nFolder structure:")
for root, dirs, files in os.walk(raw):
    depth = len(Path(root).relative_to(raw).parts)
    if depth <= 2:
        exts = dict(Counter(Path(f).suffix.lower() for f in files))
        print("  " * depth + f"{Path(root).name}/  files={len(files)} {exts}")
        for f in sorted(files)[:3]:
            print("  " * (depth + 1) + f)

xl = pd.ExcelFile(DATA / "SoundwelDatasetKey.xlsx")
print("\nSheets:", xl.sheet_names)
key = xl.parse(xl.sheet_names[0])
print("Shape:", key.shape)
print(key.dtypes)
display(key.head(10))

for c in key.columns:
    n = key[c].nunique()
    if n <= 25:
        print(f"\n{c} ({n} unique):")
        print(key[c].value_counts())

--2026-11-20 17:12:59--  https://zenodo.org/records/8252482/files/Soundwel%20Dataset%20-%20Audio%20and%20Spectrograms.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.42.113, 188.185.45.63, 137.138.121.78, ...
Connecting to zenodo.org (zenodo.org)|188.185.42.113|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 326371773 (311M) [application/octet-stream]
Saving to: ‘/kaggle/working/soundwel/soundwel.zip’

     0K ........ ........ ........ ........ 10%  898K 5m19s
 32768K ........ ........ ........ ........ 20% 4.32M 2m50s
 65536K ........ ........ ........ ........ 30% 3.57M 1m59s
 98304K ........ ........ ........ ........ 41% 3.36M 89s
131072K ........ ........ ........ ........ 51% 5.75M 64s
163840K ........ ........ ........ ........ 61% 3.50M 48s
196608K ........ ........ ........ ........ 71% 3.28M 34s
229376K ........ ........ ........ ........ 82% 3.72M 21s
262144K ........ ........ ........ ........ 92% 3.73M 8s
294912K ........ ........ ...

soundwel.zip: 326.4 MB, md5 OK


--2026-11-20 17:14:51--  https://zenodo.org/records/8252482/files/SoundwelDatasetKey.xlsx?download=1
Resolving zenodo.org (zenodo.org)... 188.185.47.192, 188.185.42.113, 188.185.51.60, ...
Connecting to zenodo.org (zenodo.org)|188.185.47.192|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1682971 (1.6M) [application/octet-stream]
Saving to: ‘/kaggle/working/soundwel/SoundwelDatasetKey.xlsx’

     0K .                                  100%  792K=2.1s

2026-11-20 17:14:54 (792 KB/s) - ‘/kaggle/working/soundwel/SoundwelDatasetKey.xlsx’ saved [1682971/1682971]



SoundwelDatasetKey.xlsx: 1.7 MB, md5 OK

Folder structure:
raw/  files=0 {}
  __MACOSX/  files=0 {}
    Soundwel Dataset - Audio and Spectrograms/  files=1 {'.png': 1}
      ._IASPIASPShortPr36Sele2isolaceM1396.wav_spec.png
  Soundwel Dataset - Audio and Spectrograms/  files=13775 {'.png': 6888, '.wav': 6887}
    ._IASPIASPShortPr36Sele2isolaceM1396.wav_spec.png
    ETHZETHZIsolationIsolationpig151.wav
    ETHZETHZIsolationIsolationpig151.wav_spec.png

Sheets: ['Sheet1']
Shape: (6887, 31)
Recording Team                  object
Recording Date          datetime64[ns]
Audio Filename                  object
Spectrogram Filename            object
Age Category                    object
Age Cat NB                     float64
Sex                             object
Context                         object
Context Category                object
Call Type (Freq)                object
Call Type NB                   float64
Valence                         object
F0Mean                         float64

,Recording Team,Recording Date,Audio Filename,Spectrogram Filename,Age Category,Age Cat NB,Sex,Context,Context Category,Call Type (Freq),...,Harmonicity,WienEntrMean,DFMin,LPC1,LPC2,LPC3,LPC4,LPC5,LPC6,Type
0,ETHZ,2014-08-25,ETHZETHZPositivePositivepig1510.wav,ETHZETHZPositivePositivepig1510.wav_spec.png,Weaner,2.0,male,Enriched,Enriched,LFs,...,-0.03,-1.421,1664.488,391.943,958.805,2149.773,3380.637,3859.119,5315.941,LF
1,ETHZ,2014-08-25,ETHZETHZPositivePositivepig1513.wav,ETHZETHZPositivePositivepig1513.wav_spec.png,Weaner,2.0,male,Enriched,Enriched,LFs,...,-0.89,-1.732,1071.457,1038.448,2195.270,3373.159,5547.741,6846.378,7874.259,LF
2,ETHZ,2014-08-25,ETHZETHZPositivePositivepig1514.wav,ETHZETHZPositivePositivepig1514.wav_spec.png,Weaner,2.0,male,Enriched,Enriched,LFs,...,-1.77,-1.277,1223.329,405.034,2224.366,3342.285,3906.131,6044.269,6868.060,LF
3,ETHZ,2014-08-25,ETHZETHZPositivePositivepig1515.wav,ETHZETHZPositivePositivepig1515.wav_spec.png,Weaner,2.0,male,Enriched,Enriched,LFm,...,1.05,-2.217,1101.899,390.746,1174.710,2434.260,3508.924,3855.396,6174.135,LF
4,ETHZ,2014-08-25,ETHZETHZPositivePositivepig1516.wav,ETHZETHZPositivePositivepig1516.wav_spec.png,Weaner,2.0,male,Enriched,Enriched,LFs,...,0.48,-1.324,891.086,369.364,1167.424,2252.133,3095.539,3849.726,4707.992,LF
5,ETHZ,2014-08-25,ETHZETHZPositivePositivepig154.wav,ETHZETHZPositivePositivepig154.wav_spec.png,Weaner,2.0,male,Enriched,Enriched,LFs,...,0.22,-1.457,950.183,400.752,1143.010,2340.624,3883.149,5097.133,6159.800,LF
6,ETHZ,2014-08-25,ETHZETHZPositivePositivepig157.wav,ETHZETHZPositivePositivepig157.wav_spec.png,Weaner,2.0,male,Enriched,Enriched,LFs,...,1.20,-1.506,1009.966,391.185,1148.065,2246.768,3335.790,3865.889,4644.518,LF
7,ETHZ,2014-08-25,ETHZETHZPositivePositivepig158.wav,ETHZETHZPositivePositivepig158.wav_spec.png,Weaner,2.0,male,Enriched,Enriched,LFs,...,0.63,-2.372,1037.249,413.956,1084.897,2340.339,3447.093,5376.911,6326.450,LF
8,ETHZ,2014-08-25,ETHZETHZPositivePositivepig159.wav,ETHZETHZPositivePositivepig159.wav_spec.png,Weaner,2.0,male,Enriched,Enriched,LFs,...,0.04,-1.381,1726.442,362.914,1024.897,2319.630,3429.013,3859.128,6353.485,LF
9,ETHZ,2014-08-26,ETHZETHZPositivePositivepig1610.wav,ETHZETHZPositivePositivepig1610.wav_spec.png,Weaner,2.0,female,Enriched,Enriched,LFs,...,0.67,-2.001,978.419,444.281,2452.255,3408.550,4160.054,4842.919,5823.054,LF



Recording Team (6 unique):
Recording Team
IASPA    2406
NMBU     1418
FBN      1349
IASPB     696
ETHZ      551
IASPC     467
Name: count, dtype: int64

Age Category (3 unique):
Age Category
Piglet         3569
Weaner         1900
GrFinishing    1418
Name: count, dtype: int64

Age Cat NB (3 unique):
Age Cat NB
1.0    3569
2.0    1900
3.0     681
Name: count, dtype: int64

Sex (2 unique):
Sex
female    3316
male      2529
Name: count, dtype: int64

Context (18 unique):
Context
LongIsolation           1553
ShortIsolation          1085
Restrain                 927
Run                      737
Barren                   585
ShortReunion             354
Enriched                 336
NovelObject              333
Castration               295
Crushing                 146
NegativeConditioning     119
PositiveConditioning     119
AfterNursing              79
Huddling                  74
Fighting                  57
MissedNursing             45
BeforeNursing             26
Surprise                 

In [3]:
import numpy as np
import soundfile as sf
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold

AUDIO_DIR = raw / "Soundwel Dataset - Audio and Spectrograms"

df = key.copy()
df.columns = [c.strip() for c in df.columns]          # headers may have trailing spaces
for c in ["Recording Team", "Audio Filename", "Spectrogram Filename", "Valence", "Context Category"]:
    df[c] = df[c].astype(str).str.strip()

# link each row to its files and check they exist
df["wav_path"] = df["Audio Filename"].map(lambda f: AUDIO_DIR / f)
df["png_path"] = df["Spectrogram Filename"].map(lambda f: AUDIO_DIR / f)
df["wav_ok"] = df["wav_path"].map(lambda p: p.exists())
df["png_ok"] = df["png_path"].map(lambda p: p.exists())
print("missing wav:", (~df.wav_ok).sum(), "| missing png:", (~df.png_ok).sum())
if (~df.wav_ok | ~df.png_ok).any():
    display(df.loc[~df.wav_ok | ~df.png_ok, ["Audio Filename", "Spectrogram Filename"]].head())
df = df[df.wav_ok & df.png_ok].reset_index(drop=True)

# label: 1 = negative (distress), 0 = positive
df["label"] = (df["Valence"] == "Neg").astype(int)

# session = one team on one day; used as the group for splitting
df["session"] = df["Recording Team"] + "_" + df["Recording Date"].astype(str)

# audio properties (reads file headers only, so it is fast)
info = [sf.info(str(p)) for p in df["wav_path"]]
df["sr"] = [i.samplerate for i in info]
df["channels"] = [i.channels for i in info]
df["duration_s"] = [i.duration for i in info]
print("\nSample rates:\n", df["sr"].value_counts())
print("\nChannels:\n", df["channels"].value_counts())
print("\nDuration (s):\n", df["duration_s"].describe().round(3))

# size of the provided spectrogram images (check a sample)
sizes = Counter(Image.open(p).size for p in df["png_path"].sample(50, random_state=0))
print("\nSpectrogram image sizes (sample of 50):", dict(sizes))

# confound checks
print("\nTeam x Valence:")
display(pd.crosstab(df["Recording Team"], df["Valence"], margins=True))
print("Context x Valence:")
display(pd.crosstab(df["Context Category"], df["Valence"]))
print("Number of sessions:", df["session"].nunique())
print("Sessions containing only one valence:",
      (df.groupby("session")["Valence"].nunique() == 1).sum())

# session-grouped, label-stratified split: ~70% train / 15% val / 15% test
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=42)
df["fold"] = -1
for k, (_, idx) in enumerate(sgkf.split(df, df["label"], groups=df["session"])):
    df.loc[idx, "fold"] = k
df["split"] = np.where(df["fold"] == 0, "test", np.where(df["fold"] == 1, "val", "train"))

print("\nSplit x Valence:")
display(pd.crosstab(df["split"], df["Valence"], margins=True))
assert df.groupby("session")["split"].nunique().max() == 1, "a session leaked across splits"
print("No session appears in more than one split.")

df.to_csv(DATA / "soundwel_meta.csv", index=False)
print("Saved", DATA / "soundwel_meta.csv")

missing wav: 0 | missing png: 0

Sample rates:
 sr
44100    6887
Name: count, dtype: int64

Channels:
 channels
1    6887
Name: count, dtype: int64

Duration (s):
 count    6887.000
mean        0.352
std         0.282
min         0.043
25%         0.181
50%         0.277
75%         0.428
max         3.595
Name: duration_s, dtype: float64

Spectrogram image sizes (sample of 50): {(875, 656): 50}

Team x Valence:


Valence,Neg,Pos,All
Recording Team,,,
ETHZ,311,240,551
FBN,1230,119,1349
IASPA,1979,427,2406
IASPB,0,696,696
IASPC,467,0,467
NMBU,585,833,1418
All,4572,2315,6887


Context x Valence:


Valence,Neg,Pos
Context Category,,
AfterNursing,0,79
Barren,585,0
BeforeNursing,0,26
Castration,295,0
Crushing,146,0
Enriched,0,336
Fighting,57,0
Huddling,0,74
Isolation,2048,0


Number of sessions: 43
Sessions containing only one valence: 22

Split x Valence:


Valence,Neg,Pos,All
split,,,
test,166,0,166
train,4111,1493,5604
val,295,822,1117
All,4572,2315,6887


No session appears in more than one split.
Saved /kaggle/working/soundwel/soundwel_meta.csv


In [11]:
from sklearn.model_selection import StratifiedKFold

N_FOLDS = 5
overall_pos = (df["label"] == 0).mean()

def grouped_folds(seed):
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    f = np.full(len(df), -1)
    for k, (_, idx) in enumerate(sgkf.split(df, df["label"], groups=df["session"])):
        f[idx] = k
    return f

def balance(f):
    # worst gap between a fold's share of positive calls and the overall share
    gaps = [abs((df["label"][f == k] == 0).mean() - overall_pos) for k in range(N_FOLDS)]
    sizes = [(f == k).sum() for k in range(N_FOLDS)]
    return max(gaps), min(sizes)

# try many seeds, keep the most balanced folds (chosen before any training)
best = min(range(500), key=lambda s: (round(balance(grouped_folds(s))[0], 3), -balance(grouped_folds(s))[1]))
df["cv_fold"] = grouped_folds(best)
gap, smallest = balance(df["cv_fold"].values)
print(f"Chosen seed {best}: worst positive-share gap {gap:.3f}, smallest fold {smallest} calls")

print("\nFold x Valence:")
display(pd.crosstab(df["cv_fold"], df["Valence"], margins=True))
print("Fold x Team:")
display(pd.crosstab(df["cv_fold"], df["Recording Team"]))
assert df.groupby("session")["cv_fold"].nunique().max() == 1, "a session leaked across folds"

# random split (ignores sessions), only used to measure leakage
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
df["random_fold"] = -1
for k, (_, idx) in enumerate(skf.split(df, df["label"])):
    df.loc[idx, "random_fold"] = k

# teams with both valences, for leave-one-team-out
mixed_teams = [t for t, g in df.groupby("Recording Team") if g["Valence"].nunique() == 2]
print("Teams usable for leave-one-team-out:", mixed_teams)

df = df.drop(columns=["fold", "split"], errors="ignore")
df.to_csv(DATA / "soundwel_meta.csv", index=False)
print("Saved", DATA / "soundwel_meta.csv")

Chosen seed 230: worst positive-share gap 0.140, smallest fold 187 calls

Fold x Valence:


Valence,Neg,Pos,All
cv_fold,,,
0,1949,741,2690
1,1152,514,1666
2,441,137,578
3,925,841,1766
4,105,82,187
All,4572,2315,6887


Fold x Team:


Recording Team,ETHZ,FBN,IASPA,IASPB,IASPC,NMBU
cv_fold,,,,,,
0,0,223,1287,696,324,160
1,0,137,1115,0,77,337
2,227,281,4,0,66,0
3,137,708,0,0,0,921
4,187,0,0,0,0,0


Teams usable for leave-one-team-out: ['ETHZ', 'FBN', 'IASPA', 'NMBU']
Saved /kaggle/working/soundwel/soundwel_meta.csv


In [12]:
rng = np.random.default_rng(0)

# one row per session: size and number of positive calls
g = df.groupby("session").agg(n=("label", "size"), pos=("label", lambda s: (s == 0).sum()))
sessions, n, pos = g.index.to_numpy(), g["n"].to_numpy(), g["pos"].to_numpy()
target_share = pos.sum() / n.sum()

def cost(assign):
    sizes = np.bincount(assign, weights=n, minlength=N_FOLDS)
    poss = np.bincount(assign, weights=pos, minlength=N_FOLDS)
    if (sizes < 800).any():                                     # every fold needs at least 800 calls
        return np.inf
    share_gap = np.abs(poss / sizes - target_share).max()      # class balance comes first
    size_gap = (sizes.max() - sizes.min()) / sizes.mean()      # size balance is a tie-breaker
    return share_gap + 0.05 * size_gap

# local search: start from random assignments, move sessions between folds while it helps
best_assign, best_cost = None, np.inf
for restart in range(200):
    a = rng.integers(0, N_FOLDS, len(sessions))
    c = cost(a)
    improved = True
    while improved:
        improved = False
        for i in rng.permutation(len(sessions)):
            for k in range(N_FOLDS):
                if k == a[i]:
                    continue
                old, a[i] = a[i], k
                c_new = cost(a)
                if c_new < c - 1e-9:
                    c, improved = c_new, True
                else:
                    a[i] = old
    if c < best_cost:
        best_cost, best_assign = c, a.copy()

df["cv_fold"] = df["session"].map(dict(zip(sessions, best_assign)))

summary = df.groupby("cv_fold").agg(calls=("label", "size"),
                                    pos_share=("label", lambda s: round((s == 0).mean(), 3)),
                                    sessions=("session", "nunique"))
print(f"Overall positive share: {target_share:.3f}")
display(summary)
print("Fold x Team:")
display(pd.crosstab(df["cv_fold"], df["Recording Team"]))
assert df.groupby("session")["cv_fold"].nunique().max() == 1, "a session leaked across folds"
print("No session appears in more than one fold.")

df.to_csv(DATA / "soundwel_meta.csv", index=False)
print("Saved", DATA / "soundwel_meta.csv")

Overall positive share: 0.336


,calls,pos_share,sessions
cv_fold,,,
0,817,0.250,11
1,1353,0.324,5
2,869,0.252,9
3,1983,0.351,2
4,1865,0.406,16


Fold x Team:


Recording Team,ETHZ,FBN,IASPA,IASPB,IASPC,NMBU
cv_fold,,,,,,
0,104,232,0,0,324,157
1,0,169,1119,0,0,65
2,447,196,0,0,0,226
3,0,0,1287,696,0,0
4,0,752,0,0,143,970


No session appears in more than one fold.
Saved /kaggle/working/soundwel/soundwel_meta.csv


In [13]:

FEATS = [
    "F0Mean", "Q25", "Q50", "Q75", "Fpeak",
    "Dur", "AMVar", "AMRate", "AMExtent",
    "Harmonicity", "WienEntrMean", "DFMin",
    "LPC1", "LPC2", "LPC3", "LPC4",
    "LPC5", "LPC6"
]

# Recreate the evaluation splits from the original notebook

cv = df["cv_fold"].to_numpy()
rf = df["random_fold"].to_numpy()
team = df["Recording Team"].to_numpy()

grouped_splits = [
    (k, cv != k, cv == k)
    for k in range(N_FOLDS)
]

random_splits = [
    (k, rf != k, rf == k)
    for k in range(N_FOLDS)
]

mixed_teams = [
    t for t, g in df.groupby("Recording Team")
    if g["Valence"].nunique() == 2
]

team_splits = [
    (t, team != t, team == t)
    for t in mixed_teams
]

print("Random folds:", len(random_splits))
print("Grouped folds:", len(grouped_splits))
print("Team folds:", len(team_splits))



Random folds: 5
Grouped folds: 5
Team folds: 4


In [15]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score

FEATS = ["F0Mean", "Q25", "Q50", "Q75", "Fpeak", "Dur", "AMVar", "AMRate", "AMExtent",
         "Harmonicity", "WienEntrMean", "DFMin", "LPC1", "LPC2", "LPC3", "LPC4", "LPC5", "LPC6"]
missing = df[FEATS].isna().sum()
print("Features with missing values:\n", missing[missing > 0] if (missing > 0).any() else "none")

X = df[FEATS].to_numpy()
y = df["label"].to_numpy()          # 1 = negative valence (distress)

def make_models():
    return {
        "LogReg": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                LogisticRegression(max_iter=2000, class_weight="balanced")),
        "HistGB": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                                 class_weight="balanced", random_state=0),
    }

def scores(y_true, prob):
    pred = (prob >= 0.5).astype(int)
    return {"macro_f1": f1_score(y_true, pred, average="macro"),
            "bal_acc": balanced_accuracy_score(y_true, pred),
            "auc": roc_auc_score(y_true, prob) if len(np.unique(y_true)) == 2 else np.nan}

def evaluate(scheme, splits):
    rows = []
    for fold_name, train, test in splits:
        for model_name, model in make_models().items():
            model.fit(X[train], y[train])
            prob = model.predict_proba(X[test])[:, 1]
            rows.append({"scheme": scheme, "fold": fold_name, "model": model_name, **scores(y[test], prob)})
    return pd.DataFrame(rows)

cv, rf, team = df["cv_fold"].to_numpy(), df["random_fold"].to_numpy(), df["Recording Team"].to_numpy()
grouped_splits = [(k, cv != k, cv == k) for k in range(N_FOLDS)]
random_splits  = [(k, rf != k, rf == k) for k in range(N_FOLDS)]
team_splits    = [(t, team != t, team == t) for t in mixed_teams]

res = pd.concat([evaluate("grouped CV", grouped_splits),
                 evaluate("random CV", random_splits),
                 evaluate("leave-one-team-out", team_splits)], ignore_index=True)

print("\nMean ± std across folds:")
display(res.groupby(["scheme", "model"])[["macro_f1", "bal_acc", "auc"]].agg(["mean", "std"]).round(3))

print("Macro-F1 per fold, grouped CV:")
display(res[res["scheme"] == "grouped CV"].pivot(index="fold", columns="model", values="macro_f1").round(3))
print("Macro-F1 per held-out team:")
display(res[res["scheme"] == "leave-one-team-out"].pivot(index="fold", columns="model", values="macro_f1").round(3))

res.to_csv(DATA / "results_acoustic_baseline.csv", index=False)

Features with missing values:
 F0Mean          3163
Q25              737
Q50              737
Q75              737
Fpeak            737
Dur              737
AMVar            817
AMRate           927
AMExtent         927
Harmonicity      737
WienEntrMean     737
DFMin           1530
LPC1            1530
LPC2            1530
LPC3            1530
LPC4            1530
LPC5            1530
LPC6            1530
dtype: int64

Mean ± std across folds:


macro_f1        bal_acc           auc       
                              mean    std    mean    std   mean    std
scheme             model                                              
grouped CV         HistGB    0.598  0.150   0.611  0.101  0.769  0.052
                   LogReg    0.535  0.132   0.558  0.114  0.689  0.066
leave-one-team-out HistGB    0.445  0.184   0.513  0.175  0.558  0.175
                   LogReg    0.442  0.192   0.482  0.198  0.574  0.220
random CV          HistGB    0.892  0.010   0.888  0.011  0.961  0.007
                   LogReg    0.735  0.018   0.773  0.018  0.810  0.010

Macro-F1 per fold, grouped CV:


model,HistGB,LogReg
fold,,
0,0.557,0.542
1,0.729,0.621
2,0.655,0.648
3,0.694,0.550
4,0.356,0.314


Macro-F1 per held-out team:


model,HistGB,LogReg
fold,,
ETHZ,0.392,0.664
FBN,0.464,0.397
IASPA,0.683,0.501
NMBU,0.241,0.205


In [17]:

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# Define our three new machine learning models
def make_our_models():
    return {
        "SVM": make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            SVC(
                kernel="rbf",
                C=1.0,
                class_weight="balanced",
                probability=True,
                random_state=42
            )
        ),

        "RandomForest": make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        ),

        "XGBoost": make_pipeline(
            SimpleImputer(strategy="median"),
            XGBClassifier(
                n_estimators=200,
                max_depth=4,
                learning_rate=0.05,
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            )
        )
    }

print("Models ready:", list(make_our_models().keys()))


Models ready: ['SVM', 'RandomForest', 'XGBoost']


In [18]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    roc_auc_score
)

def evaluate_our_models(scheme, splits):
    results = []

    for fold_name, train, test in splits:
        print(f"Running {scheme} | Fold: {fold_name}")

        for name, model in make_our_models().items():
            model.fit(X[train], y[train])

            predictions = model.predict(X[test])
            probabilities = model.predict_proba(X[test])[:, 1]

            results.append({
                "scheme": scheme,
                "fold": fold_name,
                "model": name,
                "macro_f1": f1_score(
                    y[test], predictions,
                    average="macro", zero_division=0
                ),
                "bal_acc": balanced_accuracy_score(
                    y[test], predictions
                ),
                "auc": (
                    roc_auc_score(y[test], probabilities)
                    if len(np.unique(y[test])) == 2
                    else np.nan
                )
            })

    return pd.DataFrame(results)

# Session-grouped evaluation
results_grouped = evaluate_our_models(
    "grouped CV", grouped_splits
)

# Evaluation on unseen recording teams
results_team = evaluate_our_models(
    "leave-one-team-out", team_splits
)

# Combine all results
our_results = pd.concat(
    [results_grouped, results_team],
    ignore_index=True
)

print("\nOur model performance:")
display(
    our_results.groupby(["scheme", "model"])[
        ["macro_f1", "bal_acc", "auc"]
    ].agg(["mean", "std"]).round(3)
)

# Save results
our_results.to_csv(
    "/kaggle/working/our_audio_models_results.csv",
    index=False
)


Running grouped CV | Fold: 0
Running grouped CV | Fold: 1
Running grouped CV | Fold: 2
Running grouped CV | Fold: 3
Running grouped CV | Fold: 4
Running leave-one-team-out | Fold: ETHZ
Running leave-one-team-out | Fold: FBN
Running leave-one-team-out | Fold: IASPA
Running leave-one-team-out | Fold: NMBU

Our model performance:


macro_f1        bal_acc           auc       
                                    mean    std    mean    std   mean    std
scheme             model                                                    
grouped CV         RandomForest    0.524  0.131   0.564  0.083  0.791  0.095
                   SVM             0.678  0.156   0.681  0.158  0.758  0.125
                   XGBoost         0.563  0.147   0.587  0.100  0.775  0.063
leave-one-team-out RandomForest    0.447  0.167   0.541  0.139  0.614  0.180
                   SVM             0.558  0.102   0.607  0.114  0.575  0.223
                   XGBoost         0.440  0.174   0.513  0.169  0.589  0.175

**Results and Discussion**

Three additional machine learning models—**Support Vector Machine (SVM), Random Forest, and XGBoost**—were evaluated for classifying pig vocalizations into positive and negative valence using 18 acoustic features from the Soundwel dataset.

Among the three models, **SVM achieved the highest macro-F1 score**under both evaluation strategies, scoring 0.678 in grouped cross-validation and 0.558 in leave-one-team-out evaluation. Random Forest achieved 0.524 and 0.447, while XGBoost achieved 0.563 and 0.440, respectively.

Compared with the existing HistGradientBoosting baseline, which achieved a macro-F1 of 0.445 on unseen recording teams, SVM improved the score by 0.113 (11.3 percentage points).

However, SVM's performance decreased from 0.678 to 0.558 when evaluated on unseen recording teams. This suggests that generalization across different recording environments remains challenging.

Conclusion: **SVM was the strongest of the three additional models for classifying pig vocalization valence** under the tested settings. Nevertheless, further evaluation is necessary before considering it reliable for real-world farm monitoring.